In [1]:
!pip install -q torch transformers datasets peft accelerate sentencepiece

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os, random
from pprint import pprint

RUTE_DRIVE = "/content/drive/MyDrive/transd/Translate/langs/"
print("Carpeta con los datos:", RUTE_DRIVE)

ValueError: mount failed

In [ ]:
# 2. DEFINIR LOS ARCHIVOS A USAR (MULTILINGÜE)

pairs = [
    # Español <-> Italiano
    ("ES_IT/train_clean.es", "ES_IT/train_clean.it", "spa_Latn", "ita_Latn"),
    # Español <-> Japonés
    ("ES_JA/train_clean.es", "ES_JA/train_clean.ja", "spa_Latn", "jpn_Jpan"),
    # Español <-> Chino
    ("ES_ZH/train_clean.es", "ES_ZH/train_clean.zh", "spa_Latn", "zho_Hans"),
    # Japonés <-> Chino
    ("JA_ZH/train_clean.ja", "JA_ZH/train_clean.zh", "jpn_Jpan", "zho_Hans"),
    # Italiano <-> Chino
    ("IT_ZH/train_clean.it", "IT_ZH/train_clean.zh", "ita_Latn", "zho_Hans"),
    # Italiano <-> Japonés
    ("IT_JA/train_clean.it", "IT_JA/train_clean.ja", "ita_Latn", "jpn_Jpan"),
]

MAX_LINES_PER_FILE = 50000

In [ ]:
src_sentences = []
tgt_sentences = []
src_langs = []
tgt_langs = []

for src_file, tgt_file, src_lang_code, tgt_lang_code in pairs:
    src_path = os.path.join(RUTE_DRIVE, src_file)
    tgt_path = os.path.join(RUTE_DRIVE, tgt_file)

    print(f" Leyendo pareja: {src_file} + {tgt_file}")

    with open(src_path, encoding="utf8", errors="ignore") as f:
        src_lines = f.read().splitlines()
    with open(tgt_path, encoding="utf8", errors="ignore") as f:
        tgt_lines = f.read().splitlines()

    size = min(len(src_lines), len(tgt_lines), MAX_LINES_PER_FILE)
    src_lines = src_lines[:size]
    tgt_lines = tgt_lines[:size]

    src_sentences.extend(src_lines)
    tgt_sentences.extend(tgt_lines)
    src_langs.extend([src_lang_code] * size)
    tgt_langs.extend([tgt_lang_code] * size)

    src_sentences.extend(tgt_lines)
    tgt_sentences.extend(src_lines)
    src_langs.extend([tgt_lang_code] * size)
    tgt_langs.extend([src_lang_code] * size)

print("\nTOTAL PARES (contando ambas direcciones):", len(src_sentences))

In [ ]:
# 4. CREAR DATASET HUGGINGFACE DIRECTAMENTE

from datasets import Dataset

dataset = Dataset.from_dict({
    "source": src_sentences,
    "target": tgt_sentences,
    "src_lang": src_langs,
    "tgt_lang": tgt_langs,
})

# barajamos antes de partir
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.01)
print(dataset)


In [ ]:

# 5. CONFIGURAR MODELO NLLB

MODEL_ID = "facebook/nllb-200-distilled-600M"

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [ ]:
# 6. TOKENIZACIÓN MULTILINGÜE

def preprocess(batch):
    input_ids = []
    attention_masks = []
    labels = []

    for src_text, tgt_text, s_lang, t_lang in zip(
        batch["source"], batch["target"], batch["src_lang"], batch["tgt_lang"]
    ):
        # idioma origen
        tokenizer.src_lang = s_lang
        encoded_src = tokenizer(
            src_text,
            max_length=128,
            truncation=True,
        )

        # idioma destino
        tokenizer.tgt_lang = t_lang
        encoded_tgt = tokenizer(
            text_target=tgt_text,
            max_length=128,
            truncation=True,
        )

        input_ids.append(encoded_src["input_ids"])
        attention_masks.append(encoded_src["attention_mask"])
        labels.append(encoded_tgt["input_ids"])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels,
    }

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)
print(tokenized)


In [ ]:
# 7. CARGAR MODELO + LORA

!pip install peft accelerate bitsandbytes -q

from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=32,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, peft_config)
model.to("cuda")

In [ ]:

# 8. ENTRENAMIENTO

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
os.environ["WANDB_DISABLED"] = "true"

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-6langs",
    learning_rate=2e-4,
    warmup_steps=200,
    max_grad_norm=1.0,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,

    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    num_train_epochs=1,
    max_steps=4000,

    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,

    save_strategy="no",
    logging_steps=20,

    fp16=True,

    optim="paged_adamw_8bit",

    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("\n Entrenando modelo NLLB multilingüe...\n")
trainer.train()



In [ ]:
# 9. GUARDADO FINAL

SAVE_DIR = "/content/drive/MyDrive/trand/Translate/multilangs_model/"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("\nMODELO ENTRENADO Y GUARDADO EN:")
print(SAVE_DIR)